In [ ]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError("Invalid catalog name.")

spark.conf.set("spark.sql.session.timeZone", "UTC")

refunds_source_path = (
    f"/Volumes/{catalog}/bronze/api_files/refunds"
)

dbutils.fs.mkdirs(refunds_source_path)

print("Upload refund exports to:")
print(refunds_source_path)

In [ ]:
refund_json_files = [
    file
    for file in dbutils.fs.ls(refunds_source_path)
    if file.name.endswith(".json")
]

for file in refund_json_files:
    print(f"{file.name} — {file.size} bytes")

if not refund_json_files:
    raise ValueError(
        "No refund JSON files found. Upload an export first."
    )

In [ ]:
refunds_state_path = (
    f"/Volumes/{catalog}/operations/"
    "streaming_checkpoints/refunds_api_v1"
)

refunds_schema_path = f"{refunds_state_path}/schema"
refunds_checkpoint_path = f"{refunds_state_path}/checkpoint"

refunds_target_table = f"{catalog}.bronze.refunds_raw"

In [ ]:
refunds_raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format" , "text")
    .option("cloudFiles.schemaLocation" , refunds_schema_path)
    .option("pathGlobFilter" , "*.json")
    .load(refunds_source_path)
    .select(
        F.col("value").alias("raw_json"),
        F.col("_metadata.file_path").alias("source_file"),
        F.current_timestamp().alias('ingested_at')
    )
)

In [ ]:
refunds_ingestion_query = (
    refunds_raw_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation" , refunds_checkpoint_path)
    .outputMode("append")
    .trigger(availableNow = True)
    .toTable(refunds_target_table)
)

refunds_ingestion_query.awaitTermination()

In [ ]:
saved_refunds_raw = spark.table(refunds_target_table)

display(
    saved_refunds_raw.orderBy("ingested_at", "source_file")
)

display(
    saved_refunds_raw
    .groupBy("source_file")
    .agg(F.count("*").alias("raw_records"))
    .orderBy("source_file")
)

In [ ]:
from pyspark.sql import types as T


refund_payload_schema = T.StructType([
    T.StructField("refund_id", T.StringType(), True),
    T.StructField("return_id", T.StringType(), True),
    T.StructField("payment_id", T.StringType(), True),
    T.StructField("amount", T.StringType(), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("created_at", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
])


refunds_envelope_schema = T.StructType([
    T.StructField(
        "refund_payload",
        refund_payload_schema,
        True
    ),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("extraction_id", T.StringType(), True),
    T.StructField("extracted_at", T.StringType(), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])

In [ ]:
parsed_refunds = (
    spark.table(refunds_target_table)
    .withColumn(
        "envelope",
        F.from_json(
            F.col("raw_json"),
            refunds_envelope_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record"
            }
        )
    )

    .select(
        "raw_json",
        "source_file",
        "ingested_at",

        F.col("envelope._corrupt_record")
        .alias("_corrupt_record"),

        F.col("envelope.source_system")
        .alias("source_system"),

        F.col("envelope.extraction_id")
        .alias("extraction_id"),

        F.col("envelope.extracted_at")
        .alias("extracted_at_raw"),

        F.col("envelope.refund_payload.refund_id")
        .alias("refund_id"),

        F.col("envelope.refund_payload.return_id")
        .alias("return_id"),

        F.col("envelope.refund_payload.payment_id")
        .alias("payment_id"),

        F.col("envelope.refund_payload.amount")
        .alias("amount_raw"),

        F.col("envelope.refund_payload.currency")
        .alias("currency"),

        F.col("envelope.refund_payload.status")
        .alias("status"),

        F.col("envelope.refund_payload.created_at")
        .alias("created_at_raw"),

        F.col("envelope.refund_payload.updated_at")
        .alias("updated_at_raw")
    )

    .withColumn(
        "amount",
        F.expr("try_cast(amount_raw AS DECIMAL(18, 2))")
    )

    .withColumn(
        "created_at",
        F.try_to_timestamp("created_at_raw")
    )
    .withColumn(
        "updated_at",
        F.try_to_timestamp("updated_at_raw")
    )
    .withColumn(
        "extracted_at",
        F.try_to_timestamp("extracted_at_raw")
    )
)


display(
    parsed_refunds.select(
        "refund_id",
        "return_id",
        "payment_id",
        "amount_raw",
        "amount",
        "currency",
        "status",
        "created_at",
        "updated_at",
        "_corrupt_record"
    )
)

In [ ]:
def has_text(column_name):
    return F.coalesce(
        F.length(F.trim(F.col(column_name))) > 0,
        F.lit(False)
    )


refund_rules = [
    (
        has_text("raw_json"),
        "Missing raw JSON"
    ),
    (
        F.col("_corrupt_record").isNull(),
        "Malformed JSON or incompatible JSON structure"
    ),
    (
        has_text("refund_id"),
        "Missing refund_id"
    ),
    (
        has_text("return_id"),
        "Missing return_id"
    ),
    (
        has_text("payment_id"),
        "Missing payment_id"
    ),

    # Validate the original representation before accepting conversion.
    (
        F.col("amount_raw").rlike(r"^[0-9]+(\.[0-9]{1,2})?$")
        & F.col("amount").isNotNull()
        & (F.col("amount") > 0),
        "Refund amount must be positive, fit DECIMAL(18,2), "
        "and have at most two decimal places"
    ),

    (
        F.col("currency") == "GBP",
        "Missing or unsupported refund currency"
    ),
    (
        F.col("status").isin(
            "pending",
            "succeeded",
            "failed"
        ),
        "Missing or unsupported refund status"
    ),

    (
        F.col("created_at").isNotNull(),
        "Missing or invalid created_at"
    ),
    (
        F.col("updated_at").isNotNull(),
        "Missing or invalid updated_at"
    ),
    (
        F.col("updated_at") >= F.col("created_at"),
        "updated_at must not be earlier than created_at"
    ),

    (
        F.col("source_system") == "simulated_returns_api",
        "Missing or unexpected source_system"
    ),
    (
        has_text("extraction_id"),
        "Missing extraction_id"
    ),
    (
        F.col("extracted_at").isNotNull(),
        "Missing or invalid extracted_at"
    ),
]

In [ ]:
checked_refunds = (
    parsed_refunds
    .withColumn(
        "rejection_reasons",
        F.filter(
            F.array(*[
                F.when(
                    ~F.coalesce(condition, F.lit(False)),
                    F.lit(reason)
                )
                for condition, reason in refund_rules
            ]),
            lambda reason: reason.isNotNull()
        )
    )
    .withColumn(
        "is_valid",
        F.size("rejection_reasons") == 0
    )
)


display(
    checked_refunds.select(
        "refund_id",
        "return_id",
        "payment_id",
        "amount_raw",
        "amount",
        "currency",
        "status",
        "is_valid",
        "rejection_reasons"
    )
)   

In [ ]:
valid_refunds = checked_refunds.filter(
    F.col("is_valid")
)

rejected_refunds = checked_refunds.filter(
    ~F.col("is_valid")
)

display(
    rejected_refunds.select(
        "refund_id",
        "raw_json",
        "rejection_reasons",
        "source_file"
    )
)

if rejected_refunds.limit(1).count() > 0:
    raise ValueError(
        "Refund records failed validation. "
        "Review rejected_refunds before continuing."
    )

print("Refund field validation passed.")

In [ ]:
refund_business_columns = [
    "return_id",
    "payment_id",
    "amount",
    "currency",
    "status",
    "created_at"
]

refund_version_conflicts = (
    valid_refunds
    .groupBy("refund_id", "updated_at")
    .agg(
        F.countDistinct(
            F.struct(*[
                F.col(column_name)
                for column_name in refund_business_columns
            ])
        ).alias("distinct_business_versions")
    )
    .filter(
        F.col("distinct_business_versions") > 1
    )
)

display(refund_version_conflicts)

if refund_version_conflicts.limit(1).count() > 0:
    raise ValueError(
        "Different refund states have the same refund_id "
        "and updated_at."
    )

In [ ]:
refund_identity_conflicts = (
    valid_refunds
    .groupBy("refund_id")
    .agg(
        F.countDistinct(
            F.struct(
                "return_id",
                "payment_id",
                "amount",
                "currency",
                "created_at"
            )
        ).alias("distinct_identities")
    )
    .filter(
        F.col("distinct_identities") > 1
    )
)

display(refund_identity_conflicts)

if refund_identity_conflicts.limit(1).count() > 0:
    raise ValueError(
        "A refund_id has inconsistent return, payment, amount, "
        "currency, or creation timestamp."
    )

In [ ]:
from pyspark.sql.window import Window


latest_refund_window = (
    Window
    .partitionBy("refund_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("extracted_at").desc(),
        F.col("ingested_at").desc(),
        F.col("source_file").asc(),
        F.col("raw_json").asc()
    )
)


latest_refunds = (
    valid_refunds
    .withColumn(
        "_row_number",
        F.row_number().over(latest_refund_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .select(
        "refund_id",
        "return_id",
        "payment_id",
        "amount",
        "currency",
        "status",
        "created_at",
        "updated_at",
        "source_system",
        "extraction_id",
        "extracted_at",
        "source_file",
        "ingested_at"
    )
)


display(
    latest_refunds.orderBy("refund_id")
)

In [ ]:
refund_return_reference = (
    spark.table(f"{catalog}.silver.returns_current")
    .select(
        "return_id",
        F.col("order_id").alias("return_order_id"),
        F.col("status").alias("return_status")
    )
)

refund_payment_reference = (
    spark.table(f"{catalog}.silver.payments_current")
    .select(
        "payment_id",
        F.col("order_id").alias("payment_order_id"),
        F.col("status").alias("payment_status"),
        F.col("currency").alias("payment_currency"),
        F.col("amount").alias("captured_payment_amount")
    )
)


# Reference IDs must be unique to avoid multiplying refund rows.
for reference_df, key in [
    (refund_return_reference, "return_id"),
    (refund_payment_reference, "payment_id")
]:
    duplicates = (
        reference_df
        .groupBy(key)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicates.limit(1).count() > 0:
        display(duplicates)
        raise ValueError(
            f"Duplicate {key} values in a refund reference table."
        )

In [ ]:
refunds_with_references = (
    latest_refunds

    .join(
        refund_return_reference.withColumn(
            "_return_found",
            F.lit(True)
        ),
        on="return_id",
        how="left"
    )

    .join(
        refund_payment_reference.withColumn(
            "_payment_found",
            F.lit(True)
        ),
        on="payment_id",
        how="left"
    )
)

In [ ]:
refund_reference_rules = [
    (
        F.col("_return_found").isNotNull(),
        "Referenced return does not exist"
    ),
    (
        F.col("_payment_found").isNotNull(),
        "Referenced payment does not exist"
    ),
    (
        F.col("payment_status") == "captured",
        "Referenced payment must have status captured"
    ),
    (
        F.col("captured_payment_amount").isNotNull()
        & (F.col("captured_payment_amount") > 0),
        "Referenced payment must have a positive captured amount"
    ),
    (
        F.col("return_order_id") == F.col("payment_order_id"),
        "Return and payment belong to different orders"
    ),
    (
        F.col("currency") == F.col("payment_currency"),
        "Refund currency does not match payment currency"
    ),

    # Pending and succeeded refunds require an eligible return.
    (
        (F.col("status") == "failed")
        | F.col("return_status").isin("approved", "received"),
        "Pending or succeeded refunds require an approved "
        "or received return"
    ),
]


checked_refund_references = (
    refunds_with_references
    .withColumn(
        "reference_rejection_reasons",
        F.filter(
            F.array(*[
                F.when(
                    ~F.coalesce(condition, F.lit(False)),
                    F.lit(reason)
                )
                for condition, reason in refund_reference_rules
            ]),
            lambda reason: reason.isNotNull()
        )
    )
    .withColumn(
        "references_valid",
        F.size("reference_rejection_reasons") == 0
    )
)


display(
    checked_refund_references.select(
        "refund_id",
        "return_id",
        "payment_id",
        "status",
        "return_status",
        "payment_status",
        "return_order_id",
        "payment_order_id",
        "amount",
        "captured_payment_amount",
        "references_valid",
        "reference_rejection_reasons"
    )
)


invalid_refund_references = checked_refund_references.filter(
    ~F.col("references_valid")
)

if invalid_refund_references.limit(1).count() > 0:
    raise ValueError(
        "Refund reference checks failed. "
        "Review reference_rejection_reasons before continuing."
    )

print("Refund reference checks passed.")

In [ ]:
# Pending refunds reserve capacity.
# Succeeded refunds have already consumed capacity.
# Failed refunds consume neither.
committed_refunds = checked_refund_references.filter(
    F.col("status").isin("pending", "succeeded")
)

payment_refund_totals = (
    committed_refunds
    .groupBy("payment_id", "currency")
    .agg(
        F.sum("amount").alias("committed_refund_amount")
    )
)

payment_refund_checks = (
    payment_refund_totals
    .join(
        refund_payment_reference.select(
            "payment_id",
            "captured_payment_amount"
        ),
        on="payment_id",
        how="left"
    )
    .withColumn(
        "remaining_refund_capacity",
        F.col("captured_payment_amount")
        - F.col("committed_refund_amount")
    )
)

display(payment_refund_checks)

excess_payment_refunds = payment_refund_checks.filter(
    F.col("captured_payment_amount").isNull()
    | F.col("committed_refund_amount").isNull()
    | (
        F.col("committed_refund_amount")
        > F.col("captured_payment_amount")
    )
)

if excess_payment_refunds.limit(1).count() > 0:
    display(excess_payment_refunds)
    raise ValueError(
        "Committed refunds exceed the captured payment amount "
        "or the payment limit is unavailable."
    )

In [ ]:
return_refund_limits = (
    spark.table(f"{catalog}.gold.return_details")
    .select(
        "return_id",
        F.col("currency").alias("return_currency"),
        F.col("requested_merchandise_value")
        .alias("return_merchandise_limit")
    )
)

duplicate_return_limits = (
    return_refund_limits
    .groupBy("return_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_return_limits.limit(1).count() > 0:
    display(duplicate_return_limits)
    raise ValueError(
        "More than one merchandise limit exists for a return."
    )

In [ ]:
return_refund_totals = (
    committed_refunds
    .groupBy("return_id", "currency")
    .agg(
        F.sum("amount").alias("committed_refund_amount")
    )
)

return_refund_checks = (
    return_refund_totals
    .join(
        return_refund_limits,
        on="return_id",
        how="left"
    )
    .withColumn(
        "remaining_return_capacity",
        F.col("return_merchandise_limit")
        - F.col("committed_refund_amount")
    )
)

display(return_refund_checks)

invalid_return_refund_totals = return_refund_checks.filter(
    F.col("return_merchandise_limit").isNull()
    | (F.col("return_merchandise_limit") <= 0)
    | F.col("committed_refund_amount").isNull()
    | F.col("return_currency").isNull()
    | (F.col("currency") != F.col("return_currency"))
    | (
        F.col("committed_refund_amount")
        > F.col("return_merchandise_limit")
    )
)

if invalid_return_refund_totals.limit(1).count() > 0:
    display(invalid_return_refund_totals)
    raise ValueError(
        "Refund totals exceed the return merchandise value, "
        "the return limit is unavailable, or currencies differ."
    )

print("Payment and return refund limits passed.")

In [ ]:
current_refunds = latest_refunds.select(
    "refund_id",
    "return_id",
    "payment_id",
    "amount",
    "currency",
    "status",
    "created_at",
    "updated_at",
    "source_system",
    "extraction_id",
    "extracted_at",
    "source_file",
    "ingested_at"
)

refunds_current_table = f"{catalog}.silver.refunds_current"

current_refunds.createOrReplaceTempView(
    "current_refunds_source"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {refunds_current_table}
    USING DELTA
    AS
    SELECT *
    FROM current_refunds_source
    WHERE 1 = 0
""")

In [ ]:
from delta.tables import DeltaTable


refunds_target = DeltaTable.forName(
    spark,
    refunds_current_table
)

(
    refunds_target.alias("target")
    .merge(
        current_refunds.alias("source"),
        "target.refund_id = source.refund_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved current refunds to {refunds_current_table}")

In [ ]:
saved_refunds_current = (
    spark.table(refunds_current_table)
    .select(*current_refunds.columns)
)

# Confirm one row per refund ID.
duplicate_saved_refunds = (
    saved_refunds_current
    .groupBy("refund_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_saved_refunds.limit(1).count() > 0:
    display(duplicate_saved_refunds)
    raise ValueError(
        "silver.refunds_current contains duplicate refund IDs."
    )

# Confirm the saved contents match the computed source.
missing_or_changed_refunds = current_refunds.exceptAll(
    saved_refunds_current
)

unexpected_refunds = saved_refunds_current.exceptAll(
    current_refunds
)

if (
    missing_or_changed_refunds.limit(1).count() > 0
    or unexpected_refunds.limit(1).count() > 0
):
    raise ValueError(
        "Saved refunds do not match the computed current refunds."
    )

display(
    saved_refunds_current.orderBy("refund_id")
)

print("Silver refunds verification passed.")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW
        {catalog}.gold.payment_refund_summary
    AS

    WITH refund_totals AS (
        SELECT
            payment_id,
            currency,

            COUNT(*) AS refund_count,

            SUM(
                CASE WHEN status = 'pending'
                     THEN amount ELSE 0 END
            ) AS pending_refund_amount,

            SUM(
                CASE WHEN status = 'succeeded'
                     THEN amount ELSE 0 END
            ) AS succeeded_refund_amount,

            SUM(
                CASE WHEN status = 'failed'
                     THEN amount ELSE 0 END
            ) AS failed_refund_amount

        FROM {catalog}.silver.refunds_current
        GROUP BY payment_id, currency
    ),

    payment_totals AS (
        SELECT
            p.payment_id,
            p.order_id,
            p.currency,
            p.amount AS captured_amount,

            COALESCE(r.refund_count, 0) AS refund_count,
            COALESCE(r.pending_refund_amount, 0)
                AS pending_refund_amount,
            COALESCE(r.succeeded_refund_amount, 0)
                AS succeeded_refund_amount,
            COALESCE(r.failed_refund_amount, 0)
                AS failed_refund_amount

        FROM {catalog}.silver.payments_current AS p

        LEFT JOIN refund_totals AS r
            ON p.payment_id = r.payment_id
            AND p.currency = r.currency

        WHERE p.status = 'captured'
    )

    SELECT
        *,

        captured_amount - succeeded_refund_amount
            AS net_captured_amount,

        captured_amount
            - succeeded_refund_amount
            - pending_refund_amount
            AS remaining_refund_capacity

    FROM payment_totals
""")

In [ ]:
payment_refund_summary = spark.table(
    f"{catalog}.gold.payment_refund_summary"
)

captured_payments = (
    spark.table(f"{catalog}.silver.payments_current")
    .filter(F.col("status") == "captured")
)

saved_refunds = spark.table(
    f"{catalog}.silver.refunds_current"
)

# Confirm one summary row per payment.
duplicate_summary_payments = (
    payment_refund_summary
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_summary_payments.limit(1).count() > 0:
    raise ValueError("Duplicate payments in the refund summary.")

# Confirm the view preserves every captured payment and its amount.
expected_payments = captured_payments.select(
    "payment_id",
    "order_id",
    "currency",
    F.col("amount").alias("captured_amount")
)

actual_payments = payment_refund_summary.select(
    *expected_payments.columns
)

if (
    expected_payments.exceptAll(actual_payments).limit(1).count() > 0
    or actual_payments.exceptAll(expected_payments).limit(1).count() > 0
):
    raise ValueError(
        "The refund summary does not preserve captured payments."
    )

# Ensure no saved refund is excluded by the payment/currency join.
unmatched_refunds = saved_refunds.join(
    captured_payments.select("payment_id", "currency"),
    on=["payment_id", "currency"],
    how="left_anti"
)

if unmatched_refunds.limit(1).count() > 0:
    display(unmatched_refunds)
    raise ValueError(
        "Some refunds have no matching captured payment and currency."
    )

# Refund commitments must not exceed captured money.
invalid_summary_amounts = payment_refund_summary.filter(
    F.col("net_captured_amount").isNull()
    | F.col("remaining_refund_capacity").isNull()
    | (F.col("net_captured_amount") < 0)
    | (F.col("remaining_refund_capacity") < 0)
)

if invalid_summary_amounts.limit(1).count() > 0:
    display(invalid_summary_amounts)
    raise ValueError("Invalid amounts in the payment-refund summary.")

print("Payment-refund summary checks passed.")

In [ ]:
display(
    payment_refund_summary
    .filter(F.col("payment_id") == "PAY-21a85a68eca6")
)